# Feature Death in SAEs

Walkthrough of the death mechanism on synthetic data and the centering fix.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from metrics import compute_gamma, effective_rank, geometric_median

## 1. Synthetic activations with an outlier dimension

Sparse activations over random unit-norm features, with a large constant added to one dimension — similar to the outlier dimensions in models like AlphaFold3 and ESM3.

In [2]:
torch.manual_seed(42)
np.random.seed(42)

d = 64            # activation dimension
n_features = 128  # ground-truth features (not used by SAE, just for data gen)
n_samples = 10000
outlier_magnitude = 50  # shift in dimension 0

# Random sparse features
features = np.random.randn(n_features, d)
features /= np.linalg.norm(features, axis=1, keepdims=True)

# Sparse activations: each sample uses ~5 features
coeffs = np.zeros((n_samples, n_features))
for i in range(n_samples):
    active = np.random.choice(n_features, size=5, replace=False)
    coeffs[i, active] = np.abs(np.random.randn(5))

data = coeffs @ features + np.random.randn(n_samples, d) * 0.1

# Add outlier: large mean in dimension 0
data[:, 0] += outlier_magnitude

acts = torch.tensor(data, dtype=torch.float32)
print(f"Data shape: {acts.shape}")
print(f"Mean of dim 0: {acts[:, 0].mean():.1f}")
print(f"Mean of dim 1: {acts[:, 1].mean():.2f}")

Data shape: torch.Size([10000, 64])
Mean of dim 0: 50.0
Mean of dim 1: -0.06


## 2. Outlier severity

Gamma measures how much the activation mean dominates per-token variation.

In [3]:
gamma = compute_gamma(acts.numpy())
erank = effective_rank(acts.numpy())
print(f"Gamma:          {gamma:.1f}")
print(f"Effective rank: {erank:.1f}")

Gamma:          21.5
Effective rank: 51.7


## 3. Minimal TopK SAE

The key parameter is `bias` — the vector subtracted from inputs before encoding. Standard init sets this to zero.

In [4]:
class TopKSAE(nn.Module):
    def __init__(self, d_in, d_hidden, k):
        super().__init__()
        self.k = k
        self.bias = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, d_hidden)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        with torch.no_grad():
            self.decoder.weight.data /= self.decoder.weight.data.norm(dim=0, keepdim=True)
            self.encoder.weight.data = self.decoder.weight.data.T.clone()
            nn.init.zeros_(self.encoder.bias)

    def encode_pre_topk(self, x):
        return self.encoder(x - self.bias)

    def forward(self, x):
        pre = torch.relu(self.encode_pre_topk(x))
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        sparse = torch.zeros_like(pre)
        sparse.scatter_(-1, topk_idx, topk_vals)
        return self.decoder(sparse) + self.bias

d_hidden = 256  # 4x overcompleteness
k = 5
print(f"SAE: {d} -> {d_hidden} -> {d}, k={k}")

SAE: 64 -> 256 -> 64, k=5


## 4. Dead features by pathway

**Dead-by-ReLU**: pre-activation is never positive — the feature can't fire at all.  
**Dead-by-TopK**: pre-activation goes positive, but never makes it into the top-k.

In [5]:
def count_dead(sae, acts, k):
    """Count dead features by pathway (optimized for clarity, not speed)."""
    with torch.no_grad():
        pre = sae.encode_pre_topk(acts).numpy()

    n_feat = pre.shape[1]
    post_relu = np.maximum(pre, 0)

    # Which features ever had a positive pre-activation?
    ever_positive = pre.max(axis=0) > 0

    # Which features ever made it into the top-k?
    ever_in_topk = np.zeros(n_feat, dtype=bool)
    for row in post_relu:
        topk = np.argpartition(row, -k)[-k:]
        ever_in_topk[topk[row[topk] > 0]] = True

    dead_relu = ~ever_positive
    dead_topk = ever_positive & ~ever_in_topk

    return {'relu': int(dead_relu.sum()), 'topk': int(dead_topk.sum()),
            'total': int(dead_relu.sum() + dead_topk.sum()), 'n': n_feat}

torch.manual_seed(0)
sae_no_fix = TopKSAE(d, d_hidden, k)

r = count_dead(sae_no_fix, acts, k)
print("=== Standard init ===")
print(f"Dead by ReLU:  {r['relu']:>4d} / {r['n']} ({r['relu']/r['n']*100:.1f}%)")
print(f"Dead by TopK:  {r['topk']:>4d} / {r['n']} ({r['topk']/r['n']*100:.1f}%)")
print(f"Total dead:    {r['total']:>4d} / {r['n']} ({r['total']/r['n']*100:.1f}%)")

=== Standard init ===
Dead by ReLU:   103 / 256 (40.2%)
Dead by TopK:   131 / 256 (51.2%)
Total dead:     234 / 256 (91.4%)


## 5. Fix 1: Centering

Setting the SAE bias to the geometric median of activations removes the mean shift.

In [6]:
torch.manual_seed(0)
sae_centered = TopKSAE(d, d_hidden, k)
sae_centered.bias.data = torch.from_numpy(geometric_median(acts.numpy())).float()

r2 = count_dead(sae_centered, acts, k)
print("=== With centering ===")
print(f"Dead by ReLU:  {r2['relu']:>4d} / {r2['n']} ({r2['relu']/r2['n']*100:.1f}%)")
print(f"Dead by TopK:  {r2['topk']:>4d} / {r2['n']} ({r2['topk']/r2['n']*100:.1f}%)")
print(f"Total dead:    {r2['total']:>4d} / {r2['n']} ({r2['total']/r2['n']*100:.1f}%)")

=== With centering ===
Dead by ReLU:     0 / 256 (0.0%)
Dead by TopK:     0 / 256 (0.0%)
Total dead:       0 / 256 (0.0%)


## 6. Fix 2: PCA whitening

For low effective rank activations, centering alone may not be enough. PCA whitening equalizes variance across dimensions, removing the spectral imbalance that drives TopK competition.

In this case centering already solved the problem — PCA whitening is shown here for completeness. It matters more for models like Evo1 where the activations are nearly low-rank.

In [7]:
pca = PCA(whiten=True).fit(acts.numpy())
acts_white = torch.tensor(pca.transform(acts.numpy()), dtype=torch.float32)

print(f"Effective rank before whitening: {effective_rank(acts.numpy()):.1f}")
print(f"Effective rank after whitening:  {effective_rank(acts_white.numpy()):.1f}")
print()

torch.manual_seed(0)
sae_pca = TopKSAE(d, d_hidden, k)

r3 = count_dead(sae_pca, acts_white, k)
print("=== With PCA whitening ===")
print(f"Dead by ReLU:  {r3['relu']:>4d} / {r3['n']} ({r3['relu']/r3['n']*100:.1f}%)")
print(f"Dead by TopK:  {r3['topk']:>4d} / {r3['n']} ({r3['topk']/r3['n']*100:.1f}%)")
print(f"Total dead:    {r3['total']:>4d} / {r3['n']} ({r3['total']/r3['n']*100:.1f}%)")

Effective rank before whitening: 51.7
Effective rank after whitening:  62.9



=== With PCA whitening ===
Dead by ReLU:     0 / 256 (0.0%)
Dead by TopK:     0 / 256 (0.0%)
Total dead:       0 / 256 (0.0%)


In [8]:
print(f"{'Condition':<20} {'ReLU':>8} {'TopK':>8} {'Total':>8}")
print("-" * 48)
for name, res in [("Standard init", r), ("+ Centering", r2), ("+ PCA whitening", r3)]:
    print(f"{name:<20} {res['relu']/res['n']*100:>7.1f}% {res['topk']/res['n']*100:>7.1f}% {res['total']/res['n']*100:>7.1f}%")

Condition                ReLU     TopK    Total
------------------------------------------------
Standard init           40.2%    51.2%    91.4%
+ Centering              0.0%     0.0%     0.0%
+ PCA whitening          0.0%     0.0%     0.0%
